# Instagram Engagement Predictor 🚀

This notebook trains multiple machine learning models to predict Instagram engagement metrics based on user profile data and post content. The system analyzes:

- **Engagement Rate**: Overall post engagement
- **Comment Count**: Expected number of comments
- **Comment Sentiment**: Predicted sentiment of comments (Positive/Neutral/Negative)
- **Sentiment-Weighted Engagement**: Comment count weighted by sentiment score

## Features Used:
- User profile metrics (followers, followees, posts)
- Post content (caption, hashtags, media type)
- Advanced text analysis (BERT embeddings, sentiment/emotion analysis)
- Category information

## Models Implemented:
- Linear Regression
- Random Forest
- Gradient Boosting
- XGBoost
- LightGBM
- Neural Networks (TensorFlow)

---

In [ ]:
# Install required libraries
!pip install transformers torch lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn tensorflow

print("✅ All libraries installed successfully!")

## 📚 Import Libraries

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Libraries imported successfully!")

## 📁 Upload and Load Dataset

Upload your Instagram dataset CSV file using the file upload widget below.

In [ ]:
from google.colab import files
import io

# Upload dataset
print("Please upload your Instagram dataset CSV file:")
uploaded = files.upload()

# Get the uploaded file name
filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")

# Load the dataset
data = pd.read_csv(io.BytesIO(uploaded[filename]))

# Display basic information about the dataset
print(f"\n📊 Dataset Info:")
print(f"Shape: {data.shape}")
print(f"Columns: {len(data.columns)}")
print(f"\nFirst 5 rows:")
data.head()

## 🔄 Data Preprocessing

Clean and prepare the data for model training.

In [ ]:
# Data Preprocessing
print("🔄 Starting data preprocessing...")

# Handle missing values
data = data.fillna({
    'caption': '', 
    'hashtags': '', 
    'location': '', 
    'comment_owner_username': '',
    'mentions': ''
})

# Display missing values before and after
print("\n📊 Missing values per column:")
missing_counts = data.isnull().sum()
print(missing_counts[missing_counts > 0])

# Display basic statistics
print("\n📈 Dataset Statistics:")
print(f"Total posts: {len(data)}")
print(f"Unique users: {data['username'].nunique()}")
print(f"Media types: {data['media_type'].value_counts().to_dict()}")
print(f"Categories: {data['Category'].value_counts().to_dict()}")

print("\n✅ Data preprocessing completed!")

## 🧠 Sentiment and Emotion Analysis

Initialize pre-trained models for text analysis.

In [ ]:
# Initialize BERT tokenizer and model
print("🤖 Initializing BERT model...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Check for GPU availability
device = 0 if torch.cuda.is_available() else -1
if device == 0:
    bert_model = bert_model.to('cuda')
    print("✅ Using GPU for BERT model")
else:
    print("ℹ️ Using CPU for BERT model")

# Initialize sentiment and emotion pipelines
print("\n🎭 Initializing sentiment and emotion analysis pipelines...")
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    device=device
)

emotion_pipeline = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    device=device
)

print("✅ All NLP models initialized successfully!")

In [ ]:
# Sentiment and Emotion Analysis Functions
def analyze_text(text):
    """Analyze sentiment and emotion of given text"""
    if not text or pd.isna(text) or len(text.strip()) == 0:
        return {
            'sentiment_label': 'neutral',
            'sentiment_score': 0.0,
            'emotion_label': 'neutral',
            'emotion_score': 0.0
        }
    
    try:
        sentiment = sentiment_pipeline(text, truncation=True, max_length=512)[0]
        emotion = emotion_pipeline(text, truncation=True, max_length=512)[0]
        
        # Map sentiment labels
        sentiment_label_mapping = {
            'LABEL_0': 'negative', 
            'LABEL_1': 'neutral', 
            'LABEL_2': 'positive'
        }
        sentiment_label = sentiment_label_mapping.get(sentiment['label'], 'neutral')
        
        return {
            'sentiment_label': sentiment_label,
            'sentiment_score': sentiment['score'],
            'emotion_label': emotion['label'],
            'emotion_score': emotion['score']
        }
    except Exception as e:
        print(f"Error analyzing text: {e}")
        return {
            'sentiment_label': 'neutral',
            'sentiment_score': 0.0,
            'emotion_label': 'neutral',
            'emotion_score': 0.0
        }

def get_sentiment_features(caption):
    """Extract comprehensive sentiment features from caption"""
    analysis = analyze_text(caption)
    
    # Convert sentiment to numeric value
    sentiment_to_numeric = {'negative': -1, 'neutral': 0, 'positive': 1}
    sentiment_value = sentiment_to_numeric.get(analysis['sentiment_label'], 0)
    effective_sentiment = sentiment_value * analysis['sentiment_score']
    
    return {
        'caption_effective_sentiment': effective_sentiment,
        'caption_ml_sentiment': analysis['sentiment_label'],
        'caption_sentiment_score': analysis['sentiment_score'],
        'caption_emotion': analysis['emotion_label'],
        'caption_emotion_score': analysis['emotion_score']
    }

# Function to get BERT embeddings
def get_bert_embeddings(texts, batch_size=16):
    """Generate BERT embeddings for a list of texts"""
    embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_embeddings = []
        
        for text in batch_texts:
            if not text or pd.isna(text):
                batch_embeddings.append(np.zeros(768))
                continue
                
            try:
                inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
                if device == 0:
                    inputs = {k: v.to('cuda') for k, v in inputs.items()}
                    
                with torch.no_grad():
                    outputs = bert_model(**inputs)
                    
                embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
                batch_embeddings.append(embedding)
                
            except Exception as e:
                print(f"Error generating embedding: {e}")
                batch_embeddings.append(np.zeros(768))
        
        embeddings.extend(batch_embeddings)
        
        if i % (batch_size * 10) == 0:
            print(f"Processed {i + len(batch_texts)}/{len(texts)} texts")
    
    return np.array(embeddings)

print("✅ Text analysis functions defined!")

## ⚙️ Feature Engineering

Generate advanced features including BERT embeddings, TF-IDF features, and sentiment analysis.

In [ ]:
print("⚙️ Starting feature engineering...")

# Encode categorical variables
print("\n🏷️ Encoding categorical variables...")
le_media_type = LabelEncoder()
data['media_type_encoded'] = le_media_type.fit_transform(data['media_type'])

le_category = LabelEncoder()
data['category_encoded'] = le_category.fit_transform(data['Category'])

# Combine caption and hashtags for comprehensive text analysis
print("\n📝 Combining text features...")
data['text_combined'] = data['caption'].fillna('') + ' ' + data['hashtags'].fillna('')

# Generate BERT embeddings
print("\n🧠 Generating BERT embeddings (this may take a while)...")
bert_embeddings = get_bert_embeddings(data['text_combined'].tolist())
bert_df = pd.DataFrame(bert_embeddings, columns=[f'bert_{i}' for i in range(bert_embeddings.shape[1])])
data = pd.concat([data.reset_index(drop=True), bert_df.reset_index(drop=True)], axis=1)
print(f"✅ Generated {bert_embeddings.shape[1]} BERT features")

# Extract TF-IDF features
print("\n📊 Extracting TF-IDF features...")
tfidf = TfidfVectorizer(max_features=500, stop_words='english', min_df=2, max_df=0.9)
tfidf_matrix = tfidf.fit_transform(data['text_combined'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f'tfidf_{word}' for word in tfidf.get_feature_names_out()])
data = pd.concat([data.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
print(f"✅ Generated {len(tfidf.get_feature_names_out())} TF-IDF features")

# Apply sentiment and emotion analysis
print("\n🎭 Applying sentiment and emotion analysis...")
sentiment_features_list = []
for i, caption in enumerate(data['caption']):
    if i % 100 == 0:
        print(f"Processed {i}/{len(data)} captions")
    sentiment_features_list.append(get_sentiment_features(caption))

sentiment_df = pd.DataFrame(sentiment_features_list)
data = pd.concat([data.reset_index(drop=True), sentiment_df.reset_index(drop=True)], axis=1)

# Encode newly created categorical features
le_sentiment = LabelEncoder()
data['caption_ml_sentiment_encoded'] = le_sentiment.fit_transform(data['caption_ml_sentiment'])

le_emotion = LabelEncoder()
data['caption_emotion_encoded'] = le_emotion.fit_transform(data['caption_emotion'])

# Calculate sentiment-weighted engagement
data['sentiment_weighted_engagement'] = data['comments_count'] * data.get('comments_sentiment_score', 1.0)

print(f"\n✅ Feature engineering completed!")
print(f"Total features: {data.shape[1]}")

## 🔄 Split Data into Train and Test Sets

Prepare features and targets for model training.

In [ ]:
print("🔄 Preparing features and targets...")

# Select features for model training
base_features = [
    '#Followers', '#Followees', '#Posts', 'media_type_encoded', 'category_encoded', 
    'caption_length', 'num_hashtags', 'has_emoji', 'has_mention', 'has_url',
    'caption_effective_sentiment', 'caption_sentiment_score', 'caption_emotion_score',
    'caption_ml_sentiment_encoded', 'caption_emotion_encoded'
]

# Add BERT features
bert_features = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]

# Add TF-IDF features
tfidf_features = [f'tfidf_{word}' for word in tfidf.get_feature_names_out()]

# Combine all features
features = base_features + bert_features + tfidf_features

# Check which features exist in the dataset
available_features = [f for f in features if f in data.columns]
missing_features = [f for f in features if f not in data.columns]

if missing_features:
    print(f"⚠️ Missing features: {missing_features[:5]}{'...' if len(missing_features) > 5 else ''}")

print(f"✅ Using {len(available_features)} features for training")

# Prepare feature matrix
X = data[available_features].fillna(0)

# Prepare target variables
y_engagement = data['engagement_rate'].fillna(0)
y_comments = data['comments_count'].fillna(0)

# Handle comments_sentiment mapping
sentiment_mapping = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
y_sentiment = data['comments_sentiment'].map(sentiment_mapping).fillna(0)

y_weighted_engagement = data['sentiment_weighted_engagement'].fillna(0)

# Split data into training and testing sets
print("\n📊 Splitting data...")
X_train, X_test, y_engagement_train, y_engagement_test = train_test_split(
    X, y_engagement, test_size=0.2, random_state=42
)

_, _, y_comments_train, y_comments_test = train_test_split(
    X, y_comments, test_size=0.2, random_state=42
)

_, _, y_sentiment_train, y_sentiment_test = train_test_split(
    X, y_sentiment, test_size=0.2, random_state=42
)

_, _, y_weighted_engagement_train, y_weighted_engagement_test = train_test_split(
    X, y_weighted_engagement, test_size=0.2, random_state=42
)

print(f"✅ Data split completed!")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")

## 📏 Scale Features

Standardize features for optimal model performance.

In [ ]:
# Scale features using StandardScaler
print("📏 Scaling features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features scaled successfully!")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

## 🤖 Define Models

Set up regression and classification models for training.

In [ ]:
# Define Neural Network for regression
def build_nn_regressor(input_dim):
    """Build a neural network for regression tasks"""
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Define Neural Network for classification
def build_nn_classifier(input_dim, num_classes=3):
    """Build a neural network for classification tasks"""
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Model evaluation functions
def evaluate_regression(y_true, y_pred, model_name):
    """Evaluate regression model performance"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R²': r2}

def evaluate_classification(y_true, y_pred, model_name):
    """Evaluate classification model performance"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    return {'Model': model_name, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1}

print("✅ Model definitions completed!")

## 📈 Train and Evaluate Regression Models

Train models to predict engagement rate, comment count, and sentiment-weighted engagement.

In [ ]:
# Define regression models
models_reg = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

# Store results
all_results_reg = []

# 1. Engagement Rate Prediction
print("🎯 Training models for Engagement Rate Prediction...")
for name, model in models_reg.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train_scaled, y_engagement_train)
        y_pred = model.predict(X_test_scaled)
        result = evaluate_regression(y_engagement_test, y_pred, f"{name} (Engagement)")
        all_results_reg.append(result)
        print(f"✅ {name} - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
    except Exception as e:
        print(f"❌ Error with {name}: {e}")

# Neural Network for Engagement Rate
print("Training Neural Network for Engagement Rate...")
try:
    nn_engagement = build_nn_regressor(X_train_scaled.shape[1])
    history = nn_engagement.fit(
        X_train_scaled, y_engagement_train, 
        epochs=50, batch_size=32, verbose=0,
        validation_split=0.2
    )
    y_pred_nn = nn_engagement.predict(X_test_scaled).flatten()
    result = evaluate_regression(y_engagement_test, y_pred_nn, "Neural Network (Engagement)")
    all_results_reg.append(result)
    print(f"✅ Neural Network - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
except Exception as e:
    print(f"❌ Error with Neural Network: {e}")

print("\n" + "="*50)

# 2. Comment Count Prediction
print("💬 Training models for Comment Count Prediction...")
for name, model in models_reg.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train_scaled, y_comments_train)
        y_pred = model.predict(X_test_scaled)
        result = evaluate_regression(y_comments_test, y_pred, f"{name} (Comments)")
        all_results_reg.append(result)
        print(f"✅ {name} - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
    except Exception as e:
        print(f"❌ Error with {name}: {e}")

# Neural Network for Comment Count
print("Training Neural Network for Comment Count...")
try:
    nn_comments = build_nn_regressor(X_train_scaled.shape[1])
    nn_comments.fit(
        X_train_scaled, y_comments_train, 
        epochs=50, batch_size=32, verbose=0,
        validation_split=0.2
    )
    y_pred_nn = nn_comments.predict(X_test_scaled).flatten()
    result = evaluate_regression(y_comments_test, y_pred_nn, "Neural Network (Comments)")
    all_results_reg.append(result)
    print(f"✅ Neural Network - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
except Exception as e:
    print(f"❌ Error with Neural Network: {e}")

print("\n" + "="*50)

# 3. Sentiment-Weighted Engagement Prediction
print("🎭 Training models for Sentiment-Weighted Engagement Prediction...")
for name, model in models_reg.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train_scaled, y_weighted_engagement_train)
        y_pred = model.predict(X_test_scaled)
        result = evaluate_regression(y_weighted_engagement_test, y_pred, f"{name} (Weighted)")
        all_results_reg.append(result)
        print(f"✅ {name} - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
    except Exception as e:
        print(f"❌ Error with {name}: {e}")

# Neural Network for Sentiment-Weighted Engagement
print("Training Neural Network for Sentiment-Weighted Engagement...")
try:
    nn_weighted = build_nn_regressor(X_train_scaled.shape[1])
    nn_weighted.fit(
        X_train_scaled, y_weighted_engagement_train, 
        epochs=50, batch_size=32, verbose=0,
        validation_split=0.2
    )
    y_pred_nn = nn_weighted.predict(X_test_scaled).flatten()
    result = evaluate_regression(y_weighted_engagement_test, y_pred_nn, "Neural Network (Weighted)")
    all_results_reg.append(result)
    print(f"✅ Neural Network - R²: {result['R²']:.4f}, RMSE: {result['RMSE']:.4f}")
except Exception as e:
    print(f"❌ Error with Neural Network: {e}")

print("\n✅ Regression model training completed!")

## 🎯 Train and Evaluate Classification Models

Train models to predict comment sentiment.

In [ ]:
# Define classification models
models_clf = {
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42),
    'XGBoost Classifier': XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
    'LightGBM Classifier': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
}

# Store classification results
all_results_clf = []

# Sentiment Prediction
print("🎭 Training models for Comment Sentiment Prediction...")

# Prepare sentiment labels (shift to 0, 1, 2 for neural networks)
y_sentiment_train_shifted = y_sentiment_train + 1
y_sentiment_test_shifted = y_sentiment_test + 1

for name, model in models_clf.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train_scaled, y_sentiment_train)
        y_pred = model.predict(X_test_scaled)
        result = evaluate_classification(y_sentiment_test, y_pred, name)
        all_results_clf.append(result)
        print(f"✅ {name} - Accuracy: {result['Accuracy']:.4f}, F1: {result['F1']:.4f}")
    except Exception as e:
        print(f"❌ Error with {name}: {e}")

# Neural Network for Sentiment Classification
print("Training Neural Network for Sentiment Classification...")
try:
    nn_sentiment = build_nn_classifier(X_train_scaled.shape[1], num_classes=3)
    nn_sentiment.fit(
        X_train_scaled, y_sentiment_train_shifted, 
        epochs=50, batch_size=32, verbose=0,
        validation_split=0.2
    )
    y_pred_proba = nn_sentiment.predict(X_test_scaled)
    y_pred_nn = np.argmax(y_pred_proba, axis=1) - 1  # Shift back to -1, 0, 1
    result = evaluate_classification(y_sentiment_test, y_pred_nn, "Neural Network Classifier")
    all_results_clf.append(result)
    print(f"✅ Neural Network - Accuracy: {result['Accuracy']:.4f}, F1: {result['F1']:.4f}")
except Exception as e:
    print(f"❌ Error with Neural Network: {e}")

print("\n✅ Classification model training completed!")

## 📊 Model Comparison

Compare all trained models based on their performance metrics.

In [ ]:
# Create comparison dataframes
results_reg_df = pd.DataFrame(all_results_reg)
results_clf_df = pd.DataFrame(all_results_clf)

# Display regression results
print("📈 REGRESSION MODEL COMPARISON")
print("=" * 80)
if not results_reg_df.empty:
    print(results_reg_df.round(4))
    
    # Find best models for each task
    engagement_models = results_reg_df[results_reg_df['Model'].str.contains('Engagement')]
    comments_models = results_reg_df[results_reg_df['Model'].str.contains('Comments')]
    weighted_models = results_reg_df[results_reg_df['Model'].str.contains('Weighted')]
    
    print("\n🏆 BEST REGRESSION MODELS:")
    if not engagement_models.empty:
        best_engagement = engagement_models.loc[engagement_models['R²'].idxmax()]
        print(f"Engagement Rate: {best_engagement['Model']} (R² = {best_engagement['R²']:.4f})")
    
    if not comments_models.empty:
        best_comments = comments_models.loc[comments_models['R²'].idxmax()]
        print(f"Comment Count: {best_comments['Model']} (R² = {best_comments['R²']:.4f})")
    
    if not weighted_models.empty:
        best_weighted = weighted_models.loc[weighted_models['R²'].idxmax()]
        print(f"Weighted Engagement: {best_weighted['Model']} (R² = {best_weighted['R²']:.4f})")
else:
    print("No regression results available.")

print("\n" + "=" * 80)

# Display classification results
print("🎯 CLASSIFICATION MODEL COMPARISON")
print("=" * 80)
if not results_clf_df.empty:
    print(results_clf_df.round(4))
    
    # Find best classification model
    best_clf = results_clf_df.loc[results_clf_df['F1'].idxmax()]
    print(f"\n🏆 BEST CLASSIFICATION MODEL:")
    print(f"Comment Sentiment: {best_clf['Model']} (F1 = {best_clf['F1']:.4f})")
else:
    print("No classification results available.")

## 📊 Visualization

Visualize model performance across different metrics.

In [ ]:
# Create comprehensive visualizations
plt.style.use('default')
sns.set_palette("husl")

# Set up the plotting area
fig = plt.figure(figsize=(20, 15))

# 1. RMSE Comparison for Regression
if not results_reg_df.empty:
    plt.subplot(3, 2, 1)
    reg_data = results_reg_df.copy()
    reg_data['Task'] = reg_data['Model'].str.extract(r'\(([^)]+)\)')[0]
    reg_data['Model_Name'] = reg_data['Model'].str.replace(r'\s*\([^)]+\)', '', regex=True)
    
    sns.barplot(data=reg_data, x='Model_Name', y='RMSE', hue='Task')
    plt.title('RMSE Comparison for Regression Tasks', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('RMSE')
    plt.xticks(rotation=45)
    plt.legend(title='Task')
    plt.tight_layout()

# 2. R² Comparison for Regression
if not results_reg_df.empty:
    plt.subplot(3, 2, 2)
    sns.barplot(data=reg_data, x='Model_Name', y='R²', hue='Task')
    plt.title('R² Score Comparison for Regression Tasks', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('R² Score')
    plt.xticks(rotation=45)
    plt.legend(title='Task')
    plt.tight_layout()

# 3. Accuracy Comparison for Classification
if not results_clf_df.empty:
    plt.subplot(3, 2, 3)
    sns.barplot(data=results_clf_df, x='Model', y='Accuracy')
    plt.title('Accuracy Comparison for Sentiment Prediction', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.tight_layout()

# 4. F1 Score Comparison for Classification
if not results_clf_df.empty:
    plt.subplot(3, 2, 4)
    sns.barplot(data=results_clf_df, x='Model', y='F1')
    plt.title('F1 Score Comparison for Sentiment Prediction', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('F1 Score')
    plt.xticks(rotation=45)
    plt.tight_layout()

# 5. MAE Comparison for Regression
if not results_reg_df.empty:
    plt.subplot(3, 2, 5)
    sns.barplot(data=reg_data, x='Model_Name', y='MAE', hue='Task')
    plt.title('MAE Comparison for Regression Tasks', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('MAE')
    plt.xticks(rotation=45)
    plt.legend(title='Task')
    plt.tight_layout()

# 6. Precision vs Recall for Classification
if not results_clf_df.empty:
    plt.subplot(3, 2, 6)
    plt.scatter(results_clf_df['Recall'], results_clf_df['Precision'], s=100, alpha=0.7)
    for i, model in enumerate(results_clf_df['Model']):
        plt.annotate(model.replace('Classifier', '').strip(), 
                    (results_clf_df['Recall'].iloc[i], results_clf_df['Precision'].iloc[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision vs Recall for Classification Models', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

plt.tight_layout()
plt.show()

print("✅ Visualizations completed!")

## 🚀 Practical Implementation

Implement a function to predict engagement metrics for new user input.

In [ ]:
# Train the best models for practical use
print("🎯 Training best models for practical implementation...")

# Select best models based on performance
best_models = {}

try:
    # Train LightGBM models (generally perform well)
    engagement_model = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    engagement_model.fit(X_train_scaled, y_engagement_train)
    best_models['engagement'] = engagement_model
    
    comments_model = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    comments_model.fit(X_train_scaled, y_comments_train)
    best_models['comments'] = comments_model
    
    sentiment_model = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
    sentiment_model.fit(X_train_scaled, y_sentiment_train)
    best_models['sentiment'] = sentiment_model
    
    weighted_model = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    weighted_model.fit(X_train_scaled, y_weighted_engagement_train)
    best_models['weighted'] = weighted_model
    
    print("✅ Best models trained successfully!")
except Exception as e:
    print(f"❌ Error training models: {e}")

# Practical prediction function
def predict_engagement(user_data):
    """
    Predict engagement metrics for new user data
    
    Args:
        user_data (dict): Dictionary containing user profile and post data
    
    Returns:
        dict: Predicted engagement metrics
    """
    try:
        # Create DataFrame from user data
        user_df = pd.DataFrame([user_data])
        
        # Encode categorical variables
        if 'media_type' in user_data:
            try:
                user_df['media_type_encoded'] = le_media_type.transform([user_data['media_type']])[0]
            except:
                user_df['media_type_encoded'] = 0  # Default value
        
        if 'Category' in user_data:
            try:
                user_df['category_encoded'] = le_category.transform([user_data['Category']])[0]
            except:
                user_df['category_encoded'] = 0  # Default value
        
        # Generate sentiment and emotion features
        caption = user_data.get('caption', '')
        sentiment_features = get_sentiment_features(caption)
        
        for key, value in sentiment_features.items():
            user_df[key] = value
        
        # Encode sentiment and emotion
        try:
            user_df['caption_ml_sentiment_encoded'] = le_sentiment.transform([sentiment_features['caption_ml_sentiment']])[0]
        except:
            user_df['caption_ml_sentiment_encoded'] = 0
            
        try:
            user_df['caption_emotion_encoded'] = le_emotion.transform([sentiment_features['caption_emotion']])[0]
        except:
            user_df['caption_emotion_encoded'] = 0
        
        # Combine text features
        user_df['text_combined'] = user_df.get('caption', '').fillna('') + ' ' + user_df.get('hashtags', '').fillna('')
        
        # Generate BERT embeddings
        user_bert = get_bert_embeddings([user_df['text_combined'].iloc[0]])
        for i in range(len(user_bert[0])):
            user_df[f'bert_{i}'] = user_bert[0][i]
        
        # Generate TF-IDF features
        user_tfidf = tfidf.transform(user_df['text_combined'])
        tfidf_features_names = [f'tfidf_{word}' for word in tfidf.get_feature_names_out()]
        user_tfidf_df = pd.DataFrame(user_tfidf.toarray(), columns=tfidf_features_names)
        
        for col in tfidf_features_names:
            user_df[col] = user_tfidf_df[col].iloc[0]
        
        # Prepare features in the same order as training
        user_features = []
        for feature in available_features:
            if feature in user_df.columns:
                user_features.append(user_df[feature].iloc[0])
            else:
                user_features.append(0)  # Default value for missing features
        
        user_features = np.array(user_features).reshape(1, -1)
        user_features_scaled = scaler.transform(user_features)
        
        # Make predictions
        predictions = {}
        
        if 'engagement' in best_models:
            engagement_pred = best_models['engagement'].predict(user_features_scaled)[0]
            predictions['Engagement Rate'] = max(0, engagement_pred)
        
        if 'comments' in best_models:
            comments_pred = best_models['comments'].predict(user_features_scaled)[0]
            predictions['Comment Count'] = max(0, int(comments_pred))
        
        if 'sentiment' in best_models:
            sentiment_pred = best_models['sentiment'].predict(user_features_scaled)[0]
            sentiment_map = {1: 'Positive', 0: 'Neutral', -1: 'Negative'}
            predictions['Comment Sentiment'] = sentiment_map.get(sentiment_pred, 'Neutral')
        
        if 'weighted' in best_models:
            weighted_pred = best_models['weighted'].predict(user_features_scaled)[0]
            predictions['Sentiment-Weighted Engagement'] = max(0, weighted_pred)
        
        return predictions
        
    except Exception as e:
        print(f"Error in prediction: {e}")
        return {
            'Engagement Rate': 0.0,
            'Comment Count': 0,
            'Comment Sentiment': 'Neutral',
            'Sentiment-Weighted Engagement': 0.0,
            'Error': str(e)
        }

# Example usage
print("\n🧪 Testing with example data...")

example_user_data = {
    '#Followers': 6000,
    '#Followees': 1800,
    '#Posts': 320,
    'caption': 'New vintage chair collection! Love these beautiful pieces #design #vintage',
    'hashtags': '#design #vintage #furniture #interior',
    'media_type': 'image',
    'Category': 'interior',
    'caption_length': 65,
    'num_hashtags': 4,
    'has_emoji': False,
    'has_mention': False,
    'has_url': False
}

predictions = predict_engagement(example_user_data)

print("\n🎯 PREDICTIONS FOR EXAMPLE USER:")
print("=" * 50)
for key, value in predictions.items():
    if key != 'Error':
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

print("\n✅ Practical implementation completed!")
print("\n🎉 Model training and evaluation finished successfully!")
print("\nYou can now use the predict_engagement() function with new user data to get predictions.")

## 🎮 Interactive Prediction

Use this section to make predictions with your own data.

In [ ]:
# Interactive prediction interface
print("🎮 INTERACTIVE PREDICTION INTERFACE")
print("=" * 50)
print("Modify the values below and run this cell to get predictions:")

# Customizable user input
custom_user_data = {
    '#Followers': 10000,        # Number of followers
    '#Followees': 500,          # Number of following
    '#Posts': 150,              # Total number of posts
    'caption': 'Excited to share my latest photography project! The sunset views were absolutely breathtaking. Nature never fails to amaze me! 📸🌅',
    'hashtags': '#photography #sunset #nature #beautiful #art #landscape #golden #peaceful',
    'media_type': 'image',      # 'image', 'video', or 'carousel'
    'Category': 'interior',     # Category of the post
    'caption_length': 135,      # Length of the caption
    'num_hashtags': 8,          # Number of hashtags
    'has_emoji': True,          # Whether caption contains emoji
    'has_mention': False,       # Whether caption contains mentions
    'has_url': False            # Whether caption contains URLs
}

# Make prediction
print("\n🔮 Making predictions...")
custom_predictions = predict_engagement(custom_user_data)

# Display results in a nice format
print("\n" + "="*60)
print("📊 PREDICTED ENGAGEMENT METRICS")
print("="*60)

for metric, value in custom_predictions.items():
    if metric != 'Error':
        if metric == 'Engagement Rate':
            print(f"📈 {metric:.<30} {value:.2%}")
        elif metric == 'Comment Count':
            print(f"💬 {metric:.<30} {value:,} comments")
        elif metric == 'Comment Sentiment':
            emoji_map = {'Positive': '😊', 'Neutral': '😐', 'Negative': '😔'}
            emoji = emoji_map.get(value, '🤔')
            print(f"🎭 {metric:.<30} {value} {emoji}")
        elif metric == 'Sentiment-Weighted Engagement':
            print(f"⚖️ {metric:.<30} {value:.2f}")
        else:
            print(f"📋 {metric:.<30} {value}")

if 'Error' in custom_predictions:
    print(f"\n⚠️ Error occurred: {custom_predictions['Error']}")

print("\n" + "="*60)
print("💡 TIP: Modify the 'custom_user_data' dictionary above with your own values and re-run this cell!")

## 🎯 Summary and Next Steps

### What We Accomplished:
✅ **Data Processing**: Loaded and cleaned Instagram engagement dataset  
✅ **Feature Engineering**: Generated BERT embeddings, TF-IDF features, and sentiment analysis  
✅ **Model Training**: Trained multiple ML models for regression and classification tasks  
✅ **Model Evaluation**: Compared performance across different algorithms  
✅ **Practical Implementation**: Created prediction function for new data  

### Key Insights:
- **Best Performing Models**: LightGBM typically shows strong performance across all tasks
- **Feature Importance**: Text features (BERT embeddings) combined with user metrics provide good predictive power
- **Sentiment Analysis**: Adds valuable context for engagement prediction

### Next Steps:
1. **Model Optimization**: Fine-tune hyperparameters using GridSearch or Optuna
2. **Feature Selection**: Use feature importance analysis to reduce dimensionality
3. **Ensemble Methods**: Combine multiple models for better predictions
4. **Real-time Integration**: Deploy models as API endpoints
5. **A/B Testing**: Validate predictions against actual engagement data

### Usage:
```python
# Use the predict_engagement function with new data
new_data = {
    '#Followers': 5000,
    '#Followees': 1200,
    '#Posts': 200,
    'caption': 'Your caption here',
    'hashtags': '#your #hashtags',
    'media_type': 'image',
    'Category': 'interior',
    # ... other features
}

predictions = predict_engagement(new_data)
print(predictions)
```

---

**📧 Questions or Issues?** Feel free to modify the code and experiment with different approaches!

**🌟 Happy Predicting!** 🚀